<a href="https://colab.research.google.com/github/hamshini1413/deep-learning/blob/main/7_ResNet_50_Style_Residual_Bottleneck_Block_and_Grouped_Convolutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch torchvision thop

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.profiler import profile, ProfilerActivity
from thop import profile as thop_profile

In [3]:
class DepthwiseSeparableConv(nn.Module):

    def __init__(self,in_channels,out_channels,stride=1):

        super().__init__()

        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=in_channels,
            bias=False
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self,x):

        x = self.depthwise(x)

        x = self.pointwise(x)

        x = self.bn(x)

        return F.relu(x)

In [4]:
class BottleneckBlock(nn.Module):

    expansion = 4

    def __init__(self,in_channels,channels,stride=1):

        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            channels,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.conv2 = DepthwiseSeparableConv(
            channels,
            channels,
            stride
        )

        self.conv3 = nn.Conv2d(
            channels,
            channels*self.expansion,
            kernel_size=1,
            bias=False
        )

        self.bn3 = nn.BatchNorm2d(channels*self.expansion)

        self.shortcut = nn.Sequential()

        if stride!=1 or in_channels!=channels*self.expansion:

            self.shortcut = nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    channels*self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),

                nn.BatchNorm2d(
                    channels*self.expansion
                )
            )

    def forward(self,x):

        identity = self.shortcut(x)

        out = F.relu(self.bn1(self.conv1(x)))

        out = self.conv2(out)

        out = self.bn3(self.conv3(out))

        out += identity

        out = F.relu(out)

        return out

In [5]:
model = BottleneckBlock(
    in_channels=64,
    channels=64,
    stride=1
)

print(model)

BottleneckBlock(
  (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): DepthwiseSeparableConv(
    (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
    (pointwise): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (shortcut): Sequential(
    (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)


In [6]:
x = torch.randn(1,64,56,56)

output = model(x)

print("Input Shape :",x.shape)

print("Output Shape:",output.shape)

Input Shape : torch.Size([1, 64, 56, 56])
Output Shape: torch.Size([1, 256, 56, 56])


In [7]:
total = sum(p.numel() for p in model.parameters())

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total Parameters :",total)

print("Trainable Parameters :",trainable)

Total Parameters : 42816
Trainable Parameters : 42816


In [8]:
flops,params = thop_profile(
    model,
    inputs=(x,),
    verbose=False
)

print("FLOPs :",flops)

print("Parameters :",params)

FLOPs : 138285056.0
Parameters : 42816.0


In [9]:
with profile(
    activities=[
        ProfilerActivity.CPU
    ],
    record_shapes=True
) as prof:

    model(x)

print(
    prof.key_averages().table(
        sort_by="cpu_time_total",
        row_limit=10
    )
)

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                aten::batch_norm         0.16%      40.613us        48.21%      12.117ms       3.029ms             4  
    aten::_batch_norm_impl_index         4.86%       1.220ms        48.05%      12.077ms       3.019ms             4  
         aten::native_batch_norm        42.54%      10.691ms        43.15%      10.846ms       2.711ms             4  
                    aten::conv2d         0.15%      36.946us        39.64%       9.963ms       1.993ms             5  
               aten::convolution         0.46%     114.812us        39.50%       9.927ms       1.985ms             5  
              aten::_convolution         0.32%  

In [10]:
channels = [32,64,128]

for c in channels:

    model = BottleneckBlock(
        in_channels=c,
        channels=c
    )

    dummy = torch.randn(1,c,56,56)

    flops,params = thop_profile(
        model,
        inputs=(dummy,),
        verbose=False
    )

    print("--------------------------------")

    print("Channels :",c)

    print("Parameters :",params)

    print("FLOPs :",flops)

--------------------------------
Channels : 32
Parameters : 11168.0
FLOPs : 37029888.0
--------------------------------
Channels : 64
Parameters : 42816.0
FLOPs : 138285056.0
--------------------------------
Channels : 128
Parameters : 167552.0
FLOPs : 533471232.0


In [11]:
print("=========== SUMMARY ===========")

print("Architecture : ResNet-50 Bottleneck")

print("Depthwise Convolution : Yes")

print("Grouped Convolution : Yes")

print("Projection Shortcut : Yes")

print("PyTorch Profiler : Yes")

print("FLOPs Analysis : Yes")

print("===============================")

=========== SUMMARY ===========
Architecture : ResNet-50 Bottleneck
Depthwise Convolution : Yes
Grouped Convolution : Yes
Projection Shortcut : Yes
PyTorch Profiler : Yes
FLOPs Analysis : Yes
